# 作业 3.1：HPGe $\gamma$ 能谱刻度

## 方法

先阅读 [HPGe 探测器刻度方法](../calibration_method/HpGe_Calibration_method.html)。峰形从 Gaussian signal 加局部线性背景开始，并用残差判断是否需要低能侧 step；能量刻度先采用一次函数，再根据残差判断是否需要二次项。峰中心、$\sigma$ 和峰面积分别用于能量、峰宽和效率刻度。

[ROOT 实例代码](../code/HpGe_gamma_calibration_code.html) 给出了完整分析过程，可在 **Python / PyROOT** 与 **ROOT C++** 之间切换。

## 作业

使用 [gamma.root](gamma.root) 中的 `TH1F h0` 完成 Eurica 探测阵列的能量、峰宽和效率刻度。

### 数据与实验设置

`h0` 是 $^{152}$Eu 与 $^{133}$Ba 标准源的合并能谱，横轴是尚未刻度的 pulse-height coordinate，范围为 0–2500，每个 bin 宽 0.2。数据已对每个 Euroball Cluster 的 7 个晶体进行 add-back，再合并 12 个 Cluster；本作业从这一预处理后的 histogram 开始。

Eurica 由 12 个 Euroball Cluster 组成，每个 Cluster 含 7 个 HPGe 晶体，标定源距探测器约 22 cm。

![Eurica detector array](eurica.png)

数据采集于 2013 年 2 月 13 日。源的参考日期为 1998 年 1 月 1 日；参考活度为 $^{152}$Eu 40.9 kBq（5%）和 $^{133}$Ba 42.2 kBq（3%）。记录时长为 7442 s。效率计算中先把 7442 s 当作 live time；若它实际是 wall-clock time，则还需要 dead-time correction。

### 刻度线

下表给出本数据中使用的刻度线。$P_\gamma$ 是每次核衰变发出该 gamma ray 的概率，不是源活度；同一核素的各条线共享同一个测量时活度 $A(t)$。$P_\gamma$ 在这里用于辅助判断谱线的相对显著程度，随后还将作为效率公式中的独立因子使用。

| Nuclide | $E_\gamma$ (keV) | $P_\gamma$ (%) |
| --- | ---: | ---: |
| $^{133}$Ba | 80.9979 | 34.06 |
| $^{152}$Eu | 121.7817 | 28.41 |
| $^{152}$Eu | 244.6974 | 7.55 |
| $^{133}$Ba | 276.3989 | 7.164 |
| $^{133}$Ba | 302.8508 | 18.33 |
| $^{152}$Eu | 344.2785 | 26.59 |
| $^{133}$Ba | 356.0129 | 62.05 |
| $^{152}$Eu | 778.9045 | 12.93 |
| $^{152}$Eu | 867.378 | 4.23 |
| $^{152}$Eu | 964.079 | 14.51 |
| $^{152}$Eu | 1112.076 | 13.67 |
| $^{152}$Eu | 1408.013 | 20.87 |

能量与 pulse-height coordinate 近似满足线性关系。先在 log scale 下寻找较显著的峰候选；对同一核素，可用 $P_\gamma$ 的相对大小辅助判断，对两个不同核素则还要计入各自的 $A(t)$。选择两个相隔较远且指认可靠的峰估计初步线性关系，再用它预测并核对其他刻度线的位置。观测峰强还随 full-energy peak efficiency 改变，因此 $A(t)P_\gamma$ 只用于初步定位，不能直接当作峰面积之比。

计算活度时可采用 $T_{1/2}(^{152}\mathrm{Eu})=13.517$ y、$T_{1/2}(^{133}\mathrm{Ba})=3849.3$ d。精密分析应以所用标准源证书和同一版本的 evaluated nuclear data 为准。

### 1. 能量刻度

1. 用 log scale 查看完整的 `h0`，按照上述线性关系和相对发射强度完成刻度峰的初步定位，再在候选峰附近选择局部拟合区间。峰与邻近结构重叠时应缩小或重新选择区间，而不是让背景函数吸收另一个峰。
2. 对每条刻度线拟合峰中心和误差。以 Gaussian + linear background 为起点；若低能侧本底形成明显 step，比较加入 `erfc` step 前后的峰形残差。保留能够解释残差而参数又稳定的较简单模型。
3. 将峰中心和参考能量写入 `TGraphErrors`，峰中心误差作为横坐标误差。分别拟合一次和二次刻度函数，并画出 $\Delta E=E_{\mathrm{ref}}-E_{\mathrm{cal}}$ 随 $E_{\mathrm{ref}}$ 的残差。根据残差是否有系统曲率选择刻度函数。
4. 用选定的一次刻度关系生成能量谱。保持原 histogram 的 bin 数，将横轴上下限换算为能量，并逐 bin 复制 content 和 error；检查变换前后总计数是否守恒。

### 2. 峰宽与能量分辨率

对每个刻度峰使用经残差检验的峰形模型得到 $\sigma_{ch}$，并计算

$$
\sigma_E=\left|a_1+2a_2ch\right|\sigma_{ch},
\qquad
FWHM=2.355\sigma_E.
$$

画出 FWHM–$E_\gamma$ 曲线，拟合

$$FWHM(E)=\sqrt{A+BE+CE^2},$$

并在下方给出 $FWHM_{\mathrm{data}}-FWHM_{\mathrm{fit}}$ 残差。若残差不支持某一项，采用更简单的模型。峰形拟合的系统残差也会影响 FWHM，不能只依据很小的统计拟合误差判断精度。

### 3. Full-energy peak efficiency（选做）

1. 从峰形拟合的 Gaussian signal 分量计算峰面积及其统计误差；面积误差应包含 Gaussian height 与 $\sigma$ 的 covariance。
2. 将两个源的参考活度衰变修正到测量日期，利用

   $$\varepsilon(E_\gamma)=\frac{N_{\mathrm{peak}}}{A(t)P_\gamma t_{\mathrm{live}}}$$

   计算各条刻度线的 efficiency。
3. 在 log–log 坐标上画出 efficiency–energy 曲线，用低阶 $\ln\varepsilon$–$\ln E$ 经验函数拟合，并在下方画相对残差。
4. 这些数据没有提供 dead time、true-coincidence summing 和源几何修正量，因此结果应报告为 **apparent full-energy peak efficiency**。说明哪些修正会影响其作为绝对效率使用。

源活度误差对同一核素的所有点是相关的。简单的 `TGraphErrors::Fit` 可以画出经验趋势，但不能把逐点重复加入的活度误差当作相互独立并据此解释 $\chi^2$。